# GG4257 - Creating Sustainable Cities with Spatial Data Science
## Lab 2 Challenges
All code cells will include a brief explanation of that code line, in cases where I have used websites or documentation, please refer to the [x] section of the 'references' cell at the end of the notebook to view citations.

### 220016532


# Challenge No 1:

1. Using a Dictionary, create a dataframe (table), with at least 4 columns and more than 100 rows. How come you can create this among data from scratch without defining every single row of data? 
2. Using the appropriate method, create a new DataFrame containing only the first 30 rows and the first 3 columns of the original DataFrame. Name this new DataFrame subset_df.
3. Using the appropriate method, filter the rows from the original dataframe where a numerical attribute(column) is greater than a particular numerical value, and find another categorical attribute that is equal to a specific string or text. Name this new DataFrame filtered_df.
4. Check this website https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html and apply the methods, mean, standard deviation, group_by to run fundamental statistical analysis of your created data frame.
5. Make sure you comment on your code and describe how you are manipulating the data.


In [3]:
# importing libraries
import numpy as np 
import pandas as pd


Task 1: 


In [4]:
years = range(2000, 2026) #creates a list of consecutive integers from 2000 to 2026
a = np.repeat(years, 4) #26*4 = 104 rows (uses code from lab)
b = np.random.randint(0, 40, 104) # generate random integers - 104 values between 0 and 26
c = np.random.choice (['A', 'B', 'C', 'D', 'E'], 104) # generates random categories from a-e for 104 rows as string is required for task 3
d = np.random.randint(0, 40, 104) # generate random integers - 104 values between 0 and 26

In [5]:
df = pd.DataFrame({'a':a,'b':b,'c':c,'d':d})  #the curly brackets indicate a dictionary

Task 2 :

In [7]:
subset_df = df.iloc[:30,:3] #using iloc to filter by positions and : to include all data in 30 rows and 3 columns
subset_df

,a,b,c
0,2000,33,A
1,2000,5,B
2,2000,10,E
3,2000,18,B
4,2001,23,D
5,2001,35,D
6,2001,39,C
7,2001,3,C
8,2002,22,C
9,2002,36,D


Task 3:

In [8]:
filtered_df = df[(df['b']>10) & (df['c']=='B')] #takes b as it is numerical and filters for values >10, '&' also requires column c (string) to be equal (==) to B
filtered_df

,a,b,c,d
3,2000,18,B,36
21,2005,18,B,23
39,2009,27,B,36
43,2010,28,B,23
50,2012,14,B,9
51,2012,20,B,10
63,2015,14,B,20
65,2016,39,B,8
70,2017,36,B,9
78,2019,35,B,36


Task 4:

In [17]:
df_mean = df.mean(numeric_only=True) #had to add numberic_only=True as dataframe includes strings and integers

df_std = df.std(numeric_only=True)

df_groupby= df.groupby(["c"]).mean(numeric_only=True) # groups dataframe by values in column c, and averages all columns in that group.

print(df_mean, df_std, df_groupby) # prints all methods

a    2012.500000
b      19.384615
d      22.115385
dtype: float64 a     7.536320
b    11.664657
d    11.283038
dtype: float64              a          b          d
c                                   
A  2012.714286  20.000000  23.000000
B  2013.000000  20.850000  19.250000
C  2013.857143  22.285714  22.523810
D  2009.700000  14.900000  23.250000
E  2013.090909  18.772727  22.454545


# Challenge No 2:

**Part No 1:**

1. Using the same workflow previously described, now calculate the clustered areas for the GeoPandasDataFrame `gdf_bikes_end`
2. Make sure you don't have any NaN in your columns, add a CRS, clean up the unnecessary attributes, calculate the cluster values, and plot a map of 4 calculated clusters for the return locations.


In [56]:
import requests
import geopandas as gpd

# Part 1 :

Note : gdf_bikes_end was made as part of the lab instructions, and so I am importing that code and recreating that geodataframe here.

In [68]:
# code from lab below
url_bikes = "https://api.glasgow.gov.uk/mobility/v1/get_rentals?startDate=2022-05-01&endDate=2023-05-01" #glasgow api url
response = requests.get(url_bikes) # making the query to the web server, using the Get method from the requests library 
data = response.json() # get the response from the web server, we need to translate that into a format we can manipulate, like JSON.
rental_data = data['data'] # usually, there are two labels into the web server response the metadata, and the data; we will use the data label
rental_pd = pd.DataFrame(rental_data) # creates pandas dataframe

In [69]:
nan_in_column_Lat = rental_pd['endPlaceLat'].isna().sum() # checks for total NaNs in each column, found 33
nan_in_column_Long = rental_pd['endPlaceLong'].isna().sum()
print(nan_in_column_Lat,nan_in_column_Lat) 

33 33


In [70]:
clean_rental_pd = rental_pd.dropna(subset=['startPlaceLat', 'startPlaceLong', 'endPlaceLat','endPlaceLong']) # removes NaNs in all long/lat columns

In [71]:
gdf_bikes_end = gpd.GeoDataFrame(clean_rental_pd, geometry=gpd.points_from_xy(clean_rental_pd['endPlaceLong'], clean_rental_pd['endPlaceLat'])) # creates geodataframe, using long/lat columns to create xy points.

In [72]:
gdf_bikes_end = gdf_bikes_end.set_crs("EPSG:4326") # sets crs as epsg:4326

In [73]:
# gdf_bikes_end.explore() # checking this worked, remove # to run

In [74]:
# code from lab, changed to keep cols relevant to bikes_end instead of bikes_start
keep_cols = [ #sets columns to keep in geodataframe, as not all are needed
    "endDate",
    "endPlaceId",
    "endPlaceName",
    "durationSeconds",
    "isInvalid",
    "price",
    "isEbike",
    "endPlaceLat",
    "endPlaceLong",
    "geometry",
]
gdf_bikes_end = gdf_bikes_end[keep_cols]

gdf_bikes_end.info() # shows dtypes to change in next cell.

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 9940 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   endDate          9940 non-null   object  
 1   endPlaceId       9940 non-null   object  
 2   endPlaceName     9940 non-null   object  
 3   durationSeconds  9940 non-null   int64   
 4   isInvalid        9940 non-null   bool    
 5   price            9940 non-null   float64 
 6   isEbike          9940 non-null   bool    
 7   endPlaceLat      9940 non-null   float64 
 8   endPlaceLong     9940 non-null   float64 
 9   geometry         9940 non-null   geometry
dtypes: bool(2), float64(3), geometry(1), int64(1), object(3)
memory usage: 718.3+ KB


In [75]:
gdf_bikes_end.endPlaceId = gdf_bikes_end.endPlaceId.astype(int)
gdf_bikes_end.endPlaceName = gdf_bikes_end.endPlaceName.astype(str)
gdf_bikes_end['endDate'] = pd.to_datetime(gdf_bikes_end['endDate'], format='%Y-%m-%dT%H:%M:%SZ')

In [76]:
# gdf_bikes_end.dtypes # checking no more dtypes need to be changed

### Clustering step :

In [77]:
from sklearn.cluster import KMeans
# code edited from lab
num_clusters = 4
kmeans_collection = KMeans(n_clusters=num_clusters, random_state=42)
gdf_bikes_end['kmeans_cluster'] = kmeans_collection.fit_predict(gdf_bikes_end[['endPlaceLong', 'endPlaceLat']]) # fit_predict from sklearn requires lat and long columns

In [78]:
gdf_bikes_end.head() # verify kmeans_cluster column has been created

,endDate,endPlaceId,endPlaceName,durationSeconds,isInvalid,price,isEbike,endPlaceLat,endPlaceLong,geometry,kmeans_cluster
0,2022-05-01 00:02:36,264299,ELECTRIC - Broomielaw,56,False,2.0,True,55.856600,-4.263521,POINT (-4.26352 55.85660),3
1,2022-05-01 00:22:34,3812776,Alexandra Park (south entrance) Alexandra Para...,1097,False,1.0,False,55.863128,-4.210282,POINT (-4.21028 55.86313),0
2,2022-05-01 00:10:08,264302,Botanic Gardens - ELECTRIC,344,False,1.0,False,55.878278,-4.288487,POINT (-4.28849 55.87828),1
3,2022-05-01 02:57:49,349455,ELECTRIC - Cessnock Subway Station,10400,False,6.0,False,55.851918,-4.294490,POINT (-4.29449 55.85192),1
4,2022-05-01 01:21:31,264295,ELECTRIC - St Enoch Square,4611,False,3.0,False,55.856829,-4.255292,POINT (-4.25529 55.85683),3


Note : I had an error importing leafmap, so I had to !pip install it here, if this issue occurs again please run the following cell. I'm not sure if this was an issue setting up my environment yml file.

In [79]:
# !pip install leafmap

In [ ]:
# again, code imported from lab.
import leafmap

m = leafmap.Map(center=(55.860166, -4.257505), #sets default zoom to Glasgow
                zoom=12,
                draw_control=False,
                measure_control=False,
                fullscreen_control=False,
                attribution_control=True,
                   
               )

m.add_basemap("OpenStreetMap") # I had to change basemap here, I selected OpenStreetMap from the provided list
m.add_data(
    gdf_bikes_end,
    column='kmeans_cluster', # uses previously created column
    legend_title='Clusters',
    cmap='Set1', 
    k=4,
)

#plotting map
m

OpenStreetMap has been already added before.


In [ ]:
**Part No 2:**

1. Using the Glasglow Open Data API ( Transit) https://developer.glasgow.gov.uk/api-details#api=traffic&operation=traffic-sensor-locations fetch all the sensor locations in the city.
2. Map the sensor
3. Find the WorkingZones and Calculate/Map the areas with more and fewer sensors distributed in the city.
4. You will need:
   * Get two separate Geopandas DataFrames, one for the traffic sensors and another one for the WorkingZones.
   * Using `sJoin` ( Spatial Join) https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html
   calculate the overlay of sensors and polygons.
   * Using group_by https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html to count the number of sensors per WorkingZone
   * Make sure you add the counts into the WorkingZone polygons of Glasgow so you can create a map of Zones with more and fewer traffic sensors.
   * Of course, you will need extra steps where you manipulate the data and extract what you need, for instance, clipping the Working Zones only for Glasgow.
5. Make sure you comment on your code and describe how you are manipulating the data.
